In [1]:
%pip install -q sentence-transformers

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
sentences = [
    "A neural network learns patterns from training data.",
    "Deep learning models learn representations from examples.",
    "The capital of France is Paris.",
    "Python is widely used for machine learning and AI development.",
    "A convolutional neural network is commonly used for image classification.",
    "Transformers use attention mechanisms to process relationships between tokens.",
    "The Eiffel Tower is a famous landmark in Paris.",
    "Semantic search retrieves text based on meaning rather than exact keywords.",
    "A database stores and organizes information for efficient retrieval.",
    "Gradient descent updates model parameters to reduce the loss."
]

embeddings = model.encode(sentences)

print("Embedding Shape: ", embeddings.shape)

Embedding Shape:  (10, 384)


In [4]:
query = "What does gradient descent do?"

query_embedding = model.encode(query)

print("Query embedding shape: ", query_embedding.shape)

Query embedding shape:  (384,)


In [5]:
from sentence_transformers import util

similarity_score = util.cos_sim(query_embedding, embeddings)

print("Similarity Score: ", similarity_score)

Similarity Score:  tensor([[0.3389, 0.3229, 0.0153, 0.1942, 0.2423, 0.2089, 0.0191, 0.1682, 0.1581,
         0.6401]])


In [6]:
scores = similarity_score[0]

top_k = 4
ranked_indices = scores.argsort(descending=True)[:top_k]

print("Query: ", query)
print("Top 4 results: -")

for idx in ranked_indices:
    i = idx.item()
    print(f"Score: {scores[i]:.4f} | {sentences[i]}")

Query:  What does gradient descent do?
Top 4 results: -
Score: 0.6401 | Gradient descent updates model parameters to reduce the loss.
Score: 0.3389 | A neural network learns patterns from training data.
Score: 0.3229 | Deep learning models learn representations from examples.
Score: 0.2423 | A convolutional neural network is commonly used for image classification.


In [71]:

documents = [
    {
        "title": "Gradient Descent",
        "text": """
        A machine learning model has parameters that change during training.
        A loss function measures how wrong the model's predictions are.
        Gradient descent uses gradients to determine how changes to model
        parameters affect the loss.
        It updates parameters in the direction that reduces the loss.
        The learning rate controls the size of each parameter update.
        A learning rate that is too large can cause the model to overshoot
        a good solution, while a learning rate that is too small can make
        training very slow.
        """
    },
    {
        "title": "Neural Networks",
        "text": """
        Neural networks consist of interconnected layers of artificial neurons.
        Each neuron receives numerical inputs, applies weights and a bias,
        and uses an activation function to produce an output.
        During training, neural networks learn patterns from examples by
        adjusting their weights and biases.
        The input layer receives the data, hidden layers process it, and
        the output layer produces the prediction.
        Deep learning uses neural networks with multiple hidden layers.
        These layers can learn increasingly complex patterns from data.
        """
    },
    {
        "title": "Transformers",
        "text": """
        Transformer models use attention mechanisms to represent relationships
        between tokens in a sequence.
        Attention allows a model to assign different importance to different
        tokens when processing a particular token.
        Transformers are widely used in natural language processing tasks,
        including text generation, translation, summarization, and semantic search.
        """
    }
]

In [8]:
from nltk.tokenize import sent_tokenize
import nltk

nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [72]:
chunks = []

for doc in documents:
    sentences_in_doc = sent_tokenize(doc["text"])

    for sentence in sentences_in_doc:
        chunks.append({
            "title": doc["title"],
            "text": sentence
        })

for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1} | {chunk['title']}")
    print(chunk["text"])
    print()

Chunk 1 | Gradient Descent

        A machine learning model has parameters that change during training.

Chunk 2 | Gradient Descent
A loss function measures how wrong the model's predictions are.

Chunk 3 | Gradient Descent
Gradient descent uses gradients to determine how changes to model
        parameters affect the loss.

Chunk 4 | Gradient Descent
It updates parameters in the direction that reduces the loss.

Chunk 5 | Gradient Descent
The learning rate controls the size of each parameter update.

Chunk 6 | Gradient Descent
A learning rate that is too large can cause the model to overshoot
        a good solution, while a learning rate that is too small can make
        training very slow.

Chunk 7 | Neural Networks

        Neural networks consist of interconnected layers of artificial neurons.

Chunk 8 | Neural Networks
Each neuron receives numerical inputs, applies weights and a bias,
        and uses an activation function to produce an output.

Chunk 9 | Neural Networks
Durin

In [73]:
import re

for chunk in chunks:
  chunk["text"] = re.sub(r"\s+", " ", chunk["text"]).strip()

In [74]:
chunk_texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = model.encode(chunk_texts)

print("Number of chunks:", len(chunk_texts))
print("Embedding shape:", chunk_embeddings.shape)

Number of chunks: 15
Embedding shape: (15, 384)


In [12]:
doc_query = "How does the learning rate affect training?"

doc_query_embedding = model.encode(doc_query)

print("Query embedding: ", doc_query_embedding.shape)

Query embedding:  (384,)


In [13]:
doc_similarity_score = util.cos_sim(doc_query_embedding, chunk_embeddings)

print("Similarity Score: ", doc_similarity_score)

Similarity Score:  tensor([[0.3838, 0.1939, 0.3875, 0.2349, 0.6392, 0.1504, 0.4739, 0.2869, 0.1803,
         0.1224]])


In [14]:
print(doc_similarity_score)

sorted_idx = doc_similarity_score[0].argsort(descending=True)

print("Query: ", doc_query)

for idx in sorted_idx:

  print(f"Score: {doc_similarity_score[0][idx]:.4f} | {chunk_texts[idx]}")


tensor([[0.3838, 0.1939, 0.3875, 0.2349, 0.6392, 0.1504, 0.4739, 0.2869, 0.1803,
         0.1224]])
Query:  How does the learning rate affect training?
Score: 0.6392 | The learning rate controls the size of each parameter update.
Score: 0.4739 | During training, they learn patterns from examples by adjusting their parameters.
Score: 0.3875 | Gradient descent calculates how changes to parameters affect the loss.
Score: 0.3838 | A machine learning model has parameters that change during training.
Score: 0.2869 | Deep learning uses neural networks with multiple layers.
Score: 0.2349 | It adjusts parameters using this information.
Score: 0.1939 | A loss function measures how wrong the model's predictions are.
Score: 0.1803 | Transformer models use attention mechanisms to represent relationships between tokens.
Score: 0.1504 | Neural networks consist of interconnected layers of artificial neurons.
Score: 0.1224 | They are widely used in language processing tasks, including text generation, 

In [15]:
def search_document(query, top_k=4, threshold=0.4):

  query_embedding = model.encode(query)
  similarity_score = util.cos_sim(query_embedding, chunk_embeddings)

  top_k_idx = similarity_score[0].argsort(descending=True)[:top_k]

  results = []

  for idx in top_k_idx:
        score = similarity_score[0][idx].item()

        if score < threshold:
            continue

        results.append({
            "title": chunks[idx]["title"],
            "text": chunks[idx]["text"].strip(),
            "score": score
        })

  return results


In [16]:
res = search_document("What are transformers?")

print("Query: What are transformers?")

for r in res:
  print(f"Title: {r['title']}")
  print(f"Text: {r['text']}")
  print(f"Score: {r['score']}")

Query: What are transformers?
Title: Transformers
Text: Transformer models use attention mechanisms to represent relationships between tokens.
Score: 0.43603596091270447


### Text Generation

In [17]:
%pip install -q transformers sentencepiece

In [18]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
generate_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model ready")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model ready


In [19]:
# Perfoming tokenization

prompt = "Explain how gradient descent reduces the loss during training."

inputs = tokenizer(prompt, return_tensors="pt")

outputs = generate_model.generate(
    **inputs,
    max_new_tokens=100
)

print("Raw Output: ", outputs)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Answer: ", answer)

Raw Output:  tensor([[    0, 10771,  4741, 19991,    19,     3,     9,  1573,    13,     3,
         30819,     3,     9,  2412,     5,     1]])
Answer:  Gradient descent is a method of accelerating a train.


In [20]:
query = "How does gradient descent reduce loss?"

search_results = search_document(query, 3)

context = "\n".join(
    f"Source: {search_result['title']}\nPassage: {search_result['text']}" for search_result in search_results
)

prompt = f"""Answer the question in one concise sentence.
You can use the information supported by the context.
Do not repeat ideas and do not repeat the context word to word.

Context: {context}

Question: {query}
Answer:"""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = generate_model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False,
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Query: ", query)
print("Retrived context: \n", context)
print("\nGenerated answer: \n", answer)

Query:  How does gradient descent reduce loss?
Retrived context: 
 Source: Gradient Descent
Passage: Gradient descent calculates how changes to parameters affect the loss.
Source: Gradient Descent
Passage: A loss function measures how wrong the model's predictions are.

Generated answer: 
 Gradient descent calculates how changes to parameters affect the loss.


In [21]:
query = "How does gradient descent reduce loss?"

results = search_document(query, top_k=6)

for result in results:
    print(f"Score: {result['score']:.3f}")
    print(f"Source: {result['title']}")
    print(f"Passage: {result['text']}")
    print("-" * 40)

Score: 0.851
Source: Gradient Descent
Passage: Gradient descent calculates how changes to parameters affect the loss.
----------------------------------------
Score: 0.421
Source: Gradient Descent
Passage: A loss function measures how wrong the model's predictions are.
----------------------------------------


In [22]:
query = "How do black holes evaporate?"

results = search_document(query, 4)

if not results:
    print("I don't have sufficiently relevant information to answer this question.")
else:
    for result in results:
        print(result["title"], "-", result["text"])

I don't have sufficiently relevant information to answer this question.


In [23]:
query = "How does gradient descent reduce loss?"

results = search_document(query, 4)

if not results:
    answer = "I don't have sufficiently relevant information to answer this question."
else:
    context = "\n".join(
        f"Source: {result['title']}\nPassage: {result['text']}"
        for result in results
    )

    prompt = f"""Answer the question using only the context below.
If the context does not contain the answer, say you don't know.

Context:
{context}

Question: {query}
Answer:"""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = generate_model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Question:", query)
print("Retrieved passages:", len(results))
print("Answer:", answer)
print("\nSources used:")

for result in results:
    print(f"- {result['title']}: {result['text']}")

Question: How does gradient descent reduce loss?
Retrieved passages: 2
Answer: changes to parameters affect the loss

Sources used:
- Gradient Descent: Gradient descent calculates how changes to parameters affect the loss.
- Gradient Descent: A loss function measures how wrong the model's predictions are.


In [24]:
test_cases = [
    {
        "query": "How does gradient descent reduce loss?",
        "expected_titles": ["Gradient Descent"]
    },
    {
        "query": "What controls the size of parameter updates?",
        "expected_titles": ["Gradient Descent"]
    },
    {
        "query": "What are neural networks made of?",
        "expected_titles": ["Neural Networks"]
    },
    {
        "query": "How do transformers use attention?",
        "expected_titles": ["Transformers"]
    },
    {
        "query": "How do black holes evaporate?",
        "expected_titles": []
    }
]

In [25]:
for case in test_cases:
    results = search_document(
        case["query"],
        top_k=4,
        threshold=0.40
    )

    retrieved_titles = {result["title"] for result in results}

    expected_titles = set(case["expected_titles"])

    correct = expected_titles.issubset(retrieved_titles)

    if not expected_titles:
        correct = len(results) == 0

    print("Question:", case["query"])
    print("Expected:", expected_titles)
    print("Retrieved:", retrieved_titles)
    print("Pass:", correct)
    print("-" * 50)

Question: How does gradient descent reduce loss?
Expected: {'Gradient Descent'}
Retrieved: {'Gradient Descent'}
Pass: True
--------------------------------------------------
Question: What controls the size of parameter updates?
Expected: {'Gradient Descent'}
Retrieved: {'Gradient Descent'}
Pass: True
--------------------------------------------------
Question: What are neural networks made of?
Expected: {'Neural Networks'}
Retrieved: {'Neural Networks'}
Pass: True
--------------------------------------------------
Question: How do transformers use attention?
Expected: {'Transformers'}
Retrieved: {'Transformers'}
Pass: True
--------------------------------------------------
Question: How do black holes evaporate?
Expected: set()
Retrieved: set()
Pass: True
--------------------------------------------------


In [26]:
test_cases = [
    {
        "query": "How does gradient descent reduce loss?",
        "expected_text": "Gradient descent calculates how changes to parameters affect the loss."
    },
    {
        "query": "What controls the size of parameter updates?",
        "expected_text": "The learning rate controls the size of each parameter update."
    },
    {
        "query": "What are neural networks made of?",
        "expected_text": "Neural networks consist of interconnected layers of artificial neurons."
    },
    {
        "query": "How do transformers use attention?",
        "expected_text": "Transformer models use attention mechanisms to represent relationships between tokens."
    },
    {
        "query": "How do black holes evaporate?",
        "expected_text": None
    }
]

In [27]:
for case in test_cases:
    results = search_document(
        case["query"],
        top_k=4,
        threshold=0.40
    )

    retrieved_texts = [result["text"] for result in results]

    expected_text = case["expected_text"]

    if expected_text is None:
        passed = len(results) == 0
    else:
        passed = expected_text in retrieved_texts

    print("Question:", case["query"])
    print("Expected passage:", expected_text)
    print("Retrieved passages:", len(results))
    print("Pass:", passed)
    print("-" * 60)

Question: How does gradient descent reduce loss?
Expected passage: Gradient descent calculates how changes to parameters affect the loss.
Retrieved passages: 2
Pass: True
------------------------------------------------------------
Question: What controls the size of parameter updates?
Expected passage: The learning rate controls the size of each parameter update.
Retrieved passages: 3
Pass: True
------------------------------------------------------------
Question: What are neural networks made of?
Expected passage: Neural networks consist of interconnected layers of artificial neurons.
Retrieved passages: 2
Pass: True
------------------------------------------------------------
Question: How do transformers use attention?
Expected passage: Transformer models use attention mechanisms to represent relationships between tokens.
Retrieved passages: 1
Pass: True
------------------------------------------------------------
Question: How do black holes evaporate?
Expected passage: None
Retr

In [28]:
evaluation_cases = [
    {
        "query": "How does gradient descent reduce loss?",
        "relevant_texts": [
            "Gradient descent calculates how changes to parameters affect the loss.",
            "A loss function measures how wrong the model's predictions are."
        ]
    },
    {
        "query": "What controls the size of parameter updates?",
        "relevant_texts": [
            "The learning rate controls the size of each parameter update.",
            "Gradient descent calculates how changes to parameters affect the loss."
        ]
    },
    {
        "query": "What are neural networks made of?",
        "relevant_texts": [
            "Neural networks consist of interconnected layers of artificial neurons.",
            "During training, they learn patterns from examples by adjusting their parameters."
        ]
    },
    {
        "query": "How do transformers use attention?",
        "relevant_texts": [
            "Transformer models use attention mechanisms to represent relationships between tokens."
        ]
    },
    {
        "query": "What is deep learning?",
        "relevant_texts": [
            "Deep learning uses neural networks with multiple layers.",
            "Neural networks consist of interconnected layers of artificial neurons."
        ]
    },
    {
        "query": "How do black holes evaporate?",
        "relevant_texts": []
    }
]

In [29]:
from statistics import mean

precision_scores = []
recall_scores = []

for case in evaluation_cases:
    results = search_document(
        case["query"],
        top_k=4,
        threshold=0.40
    )

    retrieved_texts = {result["text"] for result in results}
    relevant_texts = set(case["relevant_texts"])

    relevant_retrieved = retrieved_texts & relevant_texts

    if retrieved_texts:
        precision = len(relevant_retrieved) / len(retrieved_texts)
    else:
        precision = 0.0

    if relevant_texts:
        recall = len(relevant_retrieved) / len(relevant_texts)
    else:
        recall = None

    precision_scores.append(precision)

    if recall is not None:
        recall_scores.append(recall)

    print("Question:", case["query"])
    print("Retrieved passages:", len(retrieved_texts))
    print("Relevant passages retrieved:", len(relevant_retrieved))
    print(f"Precision: {precision:.2f}")
    print(
        f"Recall: {recall:.2f}"
        if recall is not None
        else "Recall: N/A (no relevant passages exist in the dataset)"
    )
    print("-" * 55)

print("Mean precision:", round(mean(precision_scores), 2))
print("Mean recall:", round(mean(recall_scores), 2))

Question: How does gradient descent reduce loss?
Retrieved passages: 2
Relevant passages retrieved: 2
Precision: 1.00
Recall: 1.00
-------------------------------------------------------
Question: What controls the size of parameter updates?
Retrieved passages: 3
Relevant passages retrieved: 1
Precision: 0.33
Recall: 0.50
-------------------------------------------------------
Question: What are neural networks made of?
Retrieved passages: 2
Relevant passages retrieved: 1
Precision: 0.50
Recall: 0.50
-------------------------------------------------------
Question: How do transformers use attention?
Retrieved passages: 1
Relevant passages retrieved: 1
Precision: 1.00
Recall: 1.00
-------------------------------------------------------
Question: What is deep learning?
Retrieved passages: 3
Relevant passages retrieved: 1
Precision: 0.33
Recall: 0.50
-------------------------------------------------------
Question: How do black holes evaporate?
Retrieved passages: 0
Relevant passages retr

In [30]:
thresholds = [0.30, 0.40, 0.50]

for threshold in thresholds:
    precision_scores = []
    recall_scores = []

    for case in evaluation_cases:
        results = search_document(
            case["query"],
            top_k=4,
            threshold=threshold
        )

        retrieved_texts = {result["text"] for result in results}
        relevant_texts = set(case["relevant_texts"])

        relevant_retrieved = retrieved_texts & relevant_texts

        precision = (
            len(relevant_retrieved) / len(retrieved_texts)
            if retrieved_texts else 0.0
        )

        recall = (
            len(relevant_retrieved) / len(relevant_texts)
            if relevant_texts else None
        )

        precision_scores.append(precision)

        if recall is not None:
            recall_scores.append(recall)

    print(f"Threshold: {threshold:.2f}")
    print(f"Mean precision: {mean(precision_scores):.2f}")
    print(f"Mean recall: {mean(recall_scores):.2f}")
    print("-" * 35)

Threshold: 0.30
Mean precision: 0.36
Mean recall: 0.90
-----------------------------------
Threshold: 0.40
Mean precision: 0.53
Mean recall: 0.70
-----------------------------------
Threshold: 0.50
Mean precision: 0.67
Mean recall: 0.60
-----------------------------------


### Adding keyword search to the pipeline

In [76]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_vectorizer = TfidfVectorizer()

tfidf_matrix = tfidf_vectorizer.fit_transform(chunk_texts)

print("Number of chunks:", len(chunks))
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))
print("TF-IDF matrix shape:", tfidf_matrix.shape)

Number of chunks: 15
Vocabulary size: 121
TF-IDF matrix shape: (15, 121)


In [32]:
def keyword_search(query, top_k=4, threshold=0.0):

  query_vector = tfidf_vectorizer.transform([query])

  score = cosine_similarity(query_vector, tfidf_matrix)[0]

  top_k_idx = score.argsort()[::-1][:top_k]

  result = []

  for idx in top_k_idx:
    i = idx.item()
    curr_score = score[i].item()

    if curr_score <= threshold:
      continue

    result.append({
      "title": chunks[i]["title"],
      "text": chunks[i]["text"].strip(),
      "score": curr_score
    })

  return result



In [33]:
result = keyword_search("What is gradient descent?", 4)

print(result)

[{'title': 'Gradient Descent', 'text': 'Gradient descent calculates how changes to parameters affect the loss.', 'score': 0.4951246503062238}]


In [34]:
comparison_queries = [
    "How does gradient descent reduce loss?",
    "What controls the size of parameter updates?",
    "What are neural networks made of?",
    "How do transformers use attention?",
    "How do black holes evaporate?"
]

for query in comparison_queries:

  semantic_result = search_document(query, 3, 0.4)
  keyword_result = keyword_search(query, 3, 0.0)

  print("\nQuestion: ", query)

  print("\nSemantic Search:")

  for result in semantic_result:
    print(f"- {result['text']} (score: {result['score']:.3f})")

  print("\nKeyword search:")
  for result in keyword_result:
      print(f"- {result['text']} (score: {result['score']:.3f})")


Question:  How does gradient descent reduce loss?

Semantic Search:
- Gradient descent calculates how changes to parameters affect the loss. (score: 0.851)
- A loss function measures how wrong the model's predictions are. (score: 0.421)

Keyword search:
- Gradient descent calculates how changes to parameters affect the loss. (score: 0.650)
- A loss function measures how wrong the model's predictions are. (score: 0.285)

Question:  What controls the size of parameter updates?

Semantic Search:
- The learning rate controls the size of each parameter update. (score: 0.705)
- It adjusts parameters using this information. (score: 0.617)
- A machine learning model has parameters that change during training. (score: 0.440)

Keyword search:
- The learning rate controls the size of each parameter update. (score: 0.758)
- Neural networks consist of interconnected layers of artificial neurons. (score: 0.232)
- A loss function measures how wrong the model's predictions are. (score: 0.098)

Questi

### Hybrid Search

In [60]:
def hybrid_search(query, top_k=4, semantic_k=10, keyword_k=10, rrf_k=60):

  semantic_result = search_document(
      query,
      semantic_k,
      0.4
  )

  keyword_result = keyword_search(
      query,
      keyword_k,
      0.0
  )

  fused_scores = {}
  passage_data = {}

  for results in [semantic_result, keyword_result]:
    for rank, result in enumerate(results, start=1):

      text = result['text']
      key = (result['title'], text)

      if key not in fused_scores:
        fused_scores[key] = 0.0
        passage_data[key] = result

      fused_scores[key] += 1 / (rrf_k+rank)

  sorted_scores = sorted(
      fused_scores,
      key=lambda x: fused_scores[x],
      reverse=True
  )

  hybrid_result = []

  for key in sorted_scores[:top_k]:
    result = passage_data[key]
    result["score"] = fused_scores[key]
    hybrid_result.append(result)

  return hybrid_result

In [45]:
query = "How does gradient descent reduce loss"

results = hybrid_search(query)

for rank, result in enumerate(results, start=1):
    print(f"{rank}. {result['text']}")
    print(f"   Source: {result['title']}")
    print(f"   RRF score: {result['score']:.4f}")

{('Gradient Descent', 'Gradient descent calculates how changes to parameters affect the loss.'): 0.03278688524590164, ('Gradient Descent', "A loss function measures how wrong the model's predictions are."): 0.03225806451612903}
1. Gradient descent calculates how changes to parameters affect the loss.
   Source: Gradient Descent
   RRF score: 0.0328
2. A loss function measures how wrong the model's predictions are.
   Source: Gradient Descent
   RRF score: 0.0323


In [46]:
query = "How does gradient descent reduce loss?"

semantic_results = search_document(
    query, top_k=10, threshold=0.4
)

keyword_results = keyword_search(
    query, top_k=10, threshold=0.0
)

print("SEMANTIC RANKING")
for rank, result in enumerate(semantic_results, start=1):
    print(f"{rank}. {result['text']}")

print("\nKEYWORD RANKING")
for rank, result in enumerate(keyword_results, start=1):
    print(f"{rank}. {result['text']}")

SEMANTIC RANKING
1. Gradient descent calculates how changes to parameters affect the loss.
2. A loss function measures how wrong the model's predictions are.

KEYWORD RANKING
1. Gradient descent calculates how changes to parameters affect the loss.
2. A loss function measures how wrong the model's predictions are.


In [57]:
hybrid_precision_scores = []
hybrid_recall_scores = []

for case in evaluation_cases:
    results = hybrid_search(
        case["query"],
        top_k=4,
    )

    retrieved_texts = {result["text"] for result in results}
    relevant_texts = set(case["relevant_texts"])

    relevant_retrieved = retrieved_texts & relevant_texts

    if retrieved_texts:
        precision = len(relevant_retrieved) / len(retrieved_texts)
    else:
        precision = 0.0

    if relevant_texts:
        recall = len(relevant_retrieved) / len(relevant_texts)
    else:
        recall = None

    hybrid_precision_scores.append(precision)

    if recall is not None:
        hybrid_recall_scores.append(recall)

    print(f"\nQuestion: {case['query']}")
    print(f"Retrieved passages: {len(retrieved_texts)}")
    print(f"Relevant passages retrieved: {len(relevant_retrieved)}")
    print(f"Precision: {precision:.2f}")

    if recall is not None:
        print(f"Recall: {recall:.2f}")
    else:
        print("Recall: N/A (no labeled relevant passages)")

print("\n--- Hybrid Search Summary ---")
print(f"Mean precision: {mean(hybrid_precision_scores):.2f}")
print(f"Mean recall: {mean(hybrid_recall_scores):.2f}")



Question: How does gradient descent reduce loss?
Retrieved passages: 2
Relevant passages retrieved: 2
Precision: 1.00
Recall: 1.00

Question: What controls the size of parameter updates?
Retrieved passages: 4
Relevant passages retrieved: 1
Precision: 0.25
Recall: 0.50

Question: What are neural networks made of?
Retrieved passages: 4
Relevant passages retrieved: 1
Precision: 0.25
Recall: 0.50

Question: How do transformers use attention?
Retrieved passages: 3
Relevant passages retrieved: 1
Precision: 0.33
Recall: 1.00

Question: What is deep learning?
Retrieved passages: 4
Relevant passages retrieved: 1
Precision: 0.25
Recall: 0.50

Question: How do black holes evaporate?
Retrieved passages: 2
Relevant passages retrieved: 0
Precision: 0.00
Recall: N/A (no labeled relevant passages)

--- Hybrid Search Summary ---
Mean precision: 0.35
Mean recall: 0.70


In [49]:

from statistics import mean

rrf_thresholds = [0.0, 0.01, 0.02, 0.025, 0.03]

for threshold in rrf_thresholds:
    precision_scores = []
    recall_scores = []

    for case in evaluation_cases:
        results = hybrid_search(
            case["query"],
            top_k=10
        )

        # Keep only passages that pass the RRF threshold
        results = [
            result for result in results
            if result["score"] >= threshold
        ]

        retrieved_texts = {result["text"] for result in results}
        relevant_texts = set(case["relevant_texts"])
        relevant_retrieved = retrieved_texts & relevant_texts

        precision = (
            len(relevant_retrieved) / len(retrieved_texts)
            if retrieved_texts else 0.0
        )

        recall = (
            len(relevant_retrieved) / len(relevant_texts)
            if relevant_texts else None
        )

        precision_scores.append(precision)

        if recall is not None:
            recall_scores.append(recall)

    print(
        f"Threshold: {threshold:.3f} | "
        f"Mean precision: {mean(precision_scores):.2f} | "
        f"Mean recall: {mean(recall_scores):.2f}"
    )


Threshold: 0.000 | Mean precision: 0.35 | Mean recall: 0.80
Threshold: 0.010 | Mean precision: 0.35 | Mean recall: 0.80
Threshold: 0.020 | Mean precision: 0.67 | Mean recall: 0.70
Threshold: 0.025 | Mean precision: 0.67 | Mean recall: 0.70
Threshold: 0.030 | Mean precision: 0.67 | Mean recall: 0.70


In [58]:
query = "How does gradient descent reduce loss?"

results = hybrid_search(query, top_k=10)

for result in results:
    print(
        f"RRF score: {result['score']:.4f} | "
        f"Title: {result['title']} | "
        f"Text: {result['text']}"
    )

RRF score: 0.0328 | Title: Gradient Descent | Text: Gradient descent calculates how changes to parameters affect the loss.
RRF score: 0.0323 | Title: Gradient Descent | Text: A loss function measures how wrong the model's predictions are.


In [61]:
def retrieve_context(query, top_k=4, rrf_threshold=0.020):
    results = hybrid_search(query, top_k=10)

    filtered_results = [
        result
        for result in results
        if result["score"] >= rrf_threshold
    ]

    return filtered_results[:top_k]

In [62]:
query = "How does gradient descent reduce loss?"

results = retrieve_context(query)

print("Question:", query)
print("Retrieved passages:", len(results))

for result in results:
    print(
        f"\nRRF score: {result['score']:.4f}"
        f"\nTitle: {result['title']}"
        f"\nText: {result['text']}"
    )

Question: How does gradient descent reduce loss?
Retrieved passages: 2

RRF score: 0.0328
Title: Gradient Descent
Text: Gradient descent calculates how changes to parameters affect the loss.

RRF score: 0.0323
Title: Gradient Descent
Text: A loss function measures how wrong the model's predictions are.


### Text generation with hybrid search

In [66]:

query = "How does gradient descent reduce loss?"

results = retrieve_context(query, top_k=4)

if not results:
    answer = "I don't have sufficiently relevant information to answer this question."

else:
    context = "\n".join(
        f"Source: {result['title']}\nPassage: {result['text']}"
        for result in results
    )

    prompt = f"""Answer the question using only the context below.
If the context does not contain the answer, say you don't know.
The answer should be generated in full.

Context:
{context}

Question: {query}
Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = generate_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

print("Question:", query)
print("Retrieved passages:", len(results))
print("Answer:", answer)

print("\nSources used:")
for result in results:
    print(f"- {result['title']}: {result['text']}")


Question: How does gradient descent reduce loss?
Retrieved passages: 2
Answer: changes to parameters affect the loss

Sources used:
- Gradient Descent: Gradient descent calculates how changes to parameters affect the loss.
- Gradient Descent: A loss function measures how wrong the model's predictions are.


In [77]:

test_queries = [
    "What is deep learning?",
    "What controls the size of parameter updates?",
    "How do black holes evaporate?"
]

for query in test_queries:
    results = retrieve_context(query, top_k=4)

    print("=" * 70)
    print("Question:", query)
    print("Retrieved passages:", len(results))

    if not results:
        answer = "I don't have sufficiently relevant information to answer this question."

    else:
        context = "\n".join(
            f"Source: {result['title']}\nPassage: {result['text']}"
            for result in results
        )

        prompt = f"""Answer the question using only the context below.
If the context does not contain the answer, say you don't know.
The answer must be in a complete sentence.

Context:
{context}

Question: {query}
Answer:"""

        inputs = tokenizer(prompt, return_tensors="pt")

        outputs = generate_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

        answer = tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )

    print("Answer:", answer)
    print("\nSources used:")

    if results:
        for result in results:
            print(f"- {result['title']}: {result['text']}")
    else:
        print("None")


Question: What is deep learning?
Retrieved passages: 2
Answer: A machine learning model

Sources used:
- Neural Networks: Deep learning uses neural networks with multiple hidden layers.
- Gradient Descent: A machine learning model has parameters that change during training.
Question: What controls the size of parameter updates?
Retrieved passages: 2
Answer: The learning rate

Sources used:
- Gradient Descent: The learning rate controls the size of each parameter update.
- Gradient Descent: It updates parameters in the direction that reduces the loss.
Question: How do black holes evaporate?
Retrieved passages: 0
Answer: I don't have sufficiently relevant information to answer this question.

Sources used:
None
